## Exercise 1

**Dataset Used:** CIFAR-10 (keras.datasets)

The following code implements the steps for this exercise. Outputs and charts are generated automatically inline.

In [ ]:
%matplotlib inline
# ============================================================
# educx GmbH – Neuronale Netze | Modul 2
# Lerntag 7: Convolutional Neural Networks (CNN)
# Niveau: Fortgeschrittene
# Aufgabe 1 von 3
# ============================================================
# Musterlösung – lauffähig in Spyder (tf_arm conda env)
# Python-Pfad: /Users/solusprime/opt/anaconda3/envs/tf_arm/bin/python
# ============================================================

import tensorflow as tf
import numpy as np
from sklearn.metrics import classification_report
import matplotlib

import matplotlib.pyplot as plt

print("TensorFlow Version:", tf.__version__)

# CIFAR-10 Klassenbezeichnungen
klassen_namen = ["Flugzeug", "Auto", "Vogel", "Katze", "Reh",
                 "Hund", "Frosch", "Pferd", "Schiff", "LKW"]

# ── 1. CIFAR-10 laden ─────────────────────────────────────────────────────────
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()
x_train = x_train.astype("float32") / 255.0
x_test  = x_test.astype("float32")  / 255.0
y_train = y_train.flatten()
y_test  = y_test.flatten()
print(f"Trainingsdaten: {x_train.shape}  Testdaten: {x_test.shape}")

# ── 2. Tiefes CNN mit Batch Normalization aufbauen ────────────────────────────
modell = tf.keras.Sequential([
    # Block 1: 32 Filter
    tf.keras.layers.Conv2D(32, (3, 3), padding="same", input_shape=(32, 32, 3)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Activation("relu"),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Dropout(0.2),

    # Block 2: 64 Filter
    tf.keras.layers.Conv2D(64, (3, 3), padding="same"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Activation("relu"),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Dropout(0.3),

    # Block 3: 128 Filter
    tf.keras.layers.Conv2D(128, (3, 3), padding="same"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Activation("relu"),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Dropout(0.4),

    # Globales Pooling statt Flatten
    tf.keras.layers.GlobalAveragePooling2D(),

    # Dichte Ausgabeschicht
    tf.keras.layers.Dense(10, activation="softmax"),
], name="CIFAR10_Tiefes_CNN")

# ── 3. Kompilieren ────────────────────────────────────────────────────────────
modell.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
modell.summary()

# ── 4. Training (20 Epochen) ──────────────────────────────────────────────────
print("\nStarte Training auf CIFAR-10 (20 Epochen)...")
callbacks = [
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_accuracy", factor=0.5, patience=3, verbose=1
    )
]
history = modell.fit(
    x_train, y_train,
    epochs=20,
    batch_size=64,
    validation_split=0.1,
    callbacks=callbacks,
    verbose=1
)

# ── 5. Evaluation und Klassifikationsbericht ──────────────────────────────────
test_loss, test_acc = modell.evaluate(x_test, y_test, verbose=0)
print(f"\nTest-Verlust:     {test_loss:.4f}")
print(f"Test-Genauigkeit: {test_acc:.4f}")

vorhersagen = np.argmax(modell.predict(x_test, verbose=0), axis=1)
print("\nKlassifikationsbericht (CIFAR-10):")
print(classification_report(y_test, vorhersagen, target_names=klassen_namen))

# ── 6. Visualisierung ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history["loss"],     label="Training-Loss")
axes[0].plot(history.history["val_loss"], label="Validierungs-Loss")
axes[0].set_title("Verlust (Loss)")
axes[0].set_xlabel("Epoche")
axes[0].set_ylabel("Verlust")
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history.history["accuracy"],     label="Training")
axes[1].plot(history.history["val_accuracy"], label="Validierung")
axes[1].set_title("Genauigkeit (Accuracy)")
axes[1].set_xlabel("Epoche")
axes[1].set_ylabel("Genauigkeit")
axes[1].legend()
axes[1].grid(True)

plt.suptitle("Tiefes CNN auf CIFAR-10 (20 Epochen)", fontsize=14)
plt.tight_layout()
plt.savefig("F7_1_cifar10_training.png", dpi=100)
plt.show()
print("Diagramm gespeichert: F7_1_cifar10_training.png")


## Exercise 2

**Dataset Used:** CIFAR-10 (keras.datasets)

The following code implements the steps for this exercise. Outputs and charts are generated automatically inline.

In [ ]:
%matplotlib inline
# ============================================================
# educx GmbH – Neuronale Netze | Modul 2
# Lerntag 7: Convolutional Neural Networks (CNN)
# Niveau: Fortgeschrittene
# Aufgabe 2 von 3
# ============================================================
# Musterlösung – lauffähig in Spyder (tf_arm conda env)
# Python-Pfad: /Users/solusprime/opt/anaconda3/envs/tf_arm/bin/python
# ============================================================

import tensorflow as tf
import numpy as np
import matplotlib

import matplotlib.pyplot as plt

print("TensorFlow Version:", tf.__version__)

# ── 1. CIFAR-10 laden ─────────────────────────────────────────────────────────
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()
x_train = x_train.astype("float32") / 255.0
x_test  = x_test.astype("float32")  / 255.0
y_train = y_train.flatten()
y_test  = y_test.flatten()

# Teilmenge für schnelleres Training
x_train_klein = x_train[:10000]
y_train_klein = y_train[:10000]
print(f"Trainingsdaten (Teilmenge): {x_train_klein.shape}")

# ── 2. Augmentierungs-Pipeline mit Keras-Schichten ───────────────────────────
augmentierung = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomContrast(0.1),
], name="Augmentierungs_Pipeline")

# ── 3. Modell-Fabrik ──────────────────────────────────────────────────────────
def cnn_erstellen(mit_augmentierung=False, name="CNN"):
    schichten = []
    if mit_augmentierung:
        # Augmentierung nur im Training aktiv
        schichten.append(augmentierung)
    schichten += [
        tf.keras.layers.Conv2D(32, (3, 3), activation="relu",
                               padding="same", input_shape=(32, 32, 3)),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Conv2D(128, (3, 3), activation="relu", padding="same"),
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dense(64, activation="relu"),
        tf.keras.layers.Dropout(0.4),
        tf.keras.layers.Dense(10, activation="softmax"),
    ]
    m = tf.keras.Sequential(schichten, name=name)
    m.compile(
        optimizer=tf.keras.optimizers.Adam(0.001),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return m

# ── 4. Modell OHNE Augmentierung ──────────────────────────────────────────────
print("\nTraining OHNE Augmentierung...")
modell_ohne = cnn_erstellen(mit_augmentierung=False, name="Ohne_Augmentierung")
history_ohne = modell_ohne.fit(
    x_train_klein, y_train_klein,
    epochs=15,
    batch_size=64,
    validation_data=(x_test, y_test),
    verbose=1
)

# ── 5. Modell MIT Augmentierung ───────────────────────────────────────────────
print("\nTraining MIT Augmentierung...")
modell_mit = cnn_erstellen(mit_augmentierung=True, name="Mit_Augmentierung")
history_mit = modell_mit.fit(
    x_train_klein, y_train_klein,
    epochs=15,
    batch_size=64,
    validation_data=(x_test, y_test),
    verbose=1
)

# ── 6. Vergleich ──────────────────────────────────────────────────────────────
acc_ohne, acc_mit = (history_ohne.history["val_accuracy"][-1],
                     history_mit.history["val_accuracy"][-1])
print(f"\nTest-Genauigkeit OHNE Augmentierung: {acc_ohne:.4f}")
print(f"Test-Genauigkeit MIT  Augmentierung: {acc_mit:.4f}")
print(f"Verbesserung durch Augmentierung:    {(acc_mit - acc_ohne)*100:+.2f}%")

# ── 7. Augmentierte Bilder visualisieren ──────────────────────────────────────
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
probe = x_train[:1]  # ein Bild

for i in range(5):
    axes[0, i].imshow(probe[0])
    axes[0, i].set_title("Original")
    axes[0, i].axis("off")

    aug_bild = augmentierung(probe, training=True)[0]
    axes[1, i].imshow(tf.clip_by_value(aug_bild, 0, 1))
    axes[1, i].set_title(f"Augmentiert #{i+1}")
    axes[1, i].axis("off")

plt.suptitle("Data Augmentation: Original vs. Augmentierte Varianten", fontsize=13)
plt.tight_layout()
plt.savefig("F7_2_augmentierung_beispiele.png", dpi=100)
plt.show()

# ── 8. Trainingsverläufe vergleichen ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(history_ohne.history["val_loss"], label="Ohne Augmentierung")
axes[0].plot(history_mit.history["val_loss"],  label="Mit Augmentierung")
axes[0].set_title("Validierungs-Verlust")
axes[0].set_xlabel("Epoche")
axes[0].set_ylabel("Verlust")
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history_ohne.history["val_accuracy"], label="Ohne Augmentierung")
axes[1].plot(history_mit.history["val_accuracy"],  label="Mit Augmentierung")
axes[1].set_title("Validierungs-Genauigkeit")
axes[1].set_xlabel("Epoche")
axes[1].set_ylabel("Genauigkeit")
axes[1].legend()
axes[1].grid(True)

plt.suptitle("Data Augmentation: Vergleich ohne vs. mit Augmentierung", fontsize=13)
plt.tight_layout()
plt.savefig("F7_2_augmentierung_vergleich.png", dpi=100)
plt.show()
print("Diagramme gespeichert: F7_2_augmentierung_*.png")


## Exercise 3

**Dataset Used:** MNIST (keras.datasets)

The following code implements the steps for this exercise. Outputs and charts are generated automatically inline.

In [ ]:
%matplotlib inline
# ============================================================
# educx GmbH – Neuronale Netze | Modul 2
# Lerntag 7: Convolutional Neural Networks (CNN)
# Niveau: Fortgeschrittene
# Aufgabe 3 von 3
# ============================================================
# Musterlösung – lauffähig in Spyder (tf_arm conda env)
# Python-Pfad: /Users/solusprime/opt/anaconda3/envs/tf_arm/bin/python
# ============================================================

import tensorflow as tf
import numpy as np
import matplotlib

import matplotlib.pyplot as plt
import time

print("TensorFlow Version:", tf.__version__)

# ── 1. MNIST-Teilmenge (2000 Samples) ────────────────────────────────────────
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train = x_train[:2000].astype("float32")[..., np.newaxis] / 255.0
y_train = y_train[:2000]
x_test  = x_test[:500].astype("float32")[..., np.newaxis]   / 255.0
y_test  = y_test[:500]
print(f"Trainingsdaten: {x_train.shape}  Testdaten: {x_test.shape}")

# ── 2. CNN-Fabrik mit variablen Hyperparametern ───────────────────────────────
def cnn_variante(filter_anzahl, kernel_groesse, pooling_typ, name):
    """Erstellt ein kleines CNN mit anpassbaren Hyperparametern."""
    if pooling_typ == "Max":
        pool_layer = tf.keras.layers.MaxPooling2D((2, 2))
    else:
        pool_layer = tf.keras.layers.AveragePooling2D((2, 2))

    modell = tf.keras.Sequential([
        tf.keras.layers.Conv2D(
            filter_anzahl, (kernel_groesse, kernel_groesse),
            activation="relu", padding="same",
            input_shape=(28, 28, 1)
        ),
        pool_layer,
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(64, activation="relu"),
        tf.keras.layers.Dense(10, activation="softmax"),
    ], name=name)

    modell.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return modell

# ── 3. Hyperparameter-Kombinationen definieren ────────────────────────────────
kombinationen = [
    {"filter": 16, "kernel": 3, "pooling": "Max"},
    {"filter": 32, "kernel": 3, "pooling": "Max"},
    {"filter": 64, "kernel": 3, "pooling": "Max"},
    {"filter": 16, "kernel": 5, "pooling": "Max"},
    {"filter": 32, "kernel": 5, "pooling": "Max"},
    {"filter": 32, "kernel": 3, "pooling": "Average"},
    {"filter": 64, "kernel": 3, "pooling": "Average"},
]

# ── 4. Alle Varianten trainieren und vergleichen ──────────────────────────────
ergebnisse = []
verlauf_alle = {}

for i, k in enumerate(kombinationen):
    name = f"F{k['filter']}_K{k['kernel']}_P{k['pooling']}"
    print(f"\n[{i+1}/{len(kombinationen)}] Trainiere: {name}")
    m = cnn_variante(k["filter"], k["kernel"], k["pooling"], name)

    start = time.perf_counter()
    history = m.fit(
        x_train, y_train,
        epochs=10,
        batch_size=32,
        validation_data=(x_test, y_test),
        verbose=0
    )
    trainingszeit = time.perf_counter() - start

    test_loss, test_acc = m.evaluate(x_test, y_test, verbose=0)
    params = m.count_params()

    ergebnisse.append({
        "Name":       name,
        "Filter":     k["filter"],
        "Kernel":     k["kernel"],
        "Pooling":    k["pooling"],
        "Params":     params,
        "Test-Acc":   test_acc,
        "Zeit (s)":   round(trainingszeit, 2),
    })
    verlauf_alle[name] = history.history["val_accuracy"]
    print(f"  → Test-Acc: {test_acc:.4f} | Params: {params:,} | Zeit: {trainingszeit:.1f}s")

# ── 5. Ergebnistabelle ausgeben ───────────────────────────────────────────────
print("\n── Vergleichstabelle ──")
header = f"{'Name':<28} {'Filter':>6} {'Kernel':>6} {'Pool':>8} {'Params':>8} {'Acc':>8} {'Zeit':>8}"
print(header)
print("-" * len(header))
for r in sorted(ergebnisse, key=lambda x: x["Test-Acc"], reverse=True):
    print(f"{r['Name']:<28} {r['Filter']:>6} {r['Kernel']:>6} "
          f"{r['Pooling']:>8} {r['Params']:>8,} {r['Test-Acc']:>8.4f} {r['Zeit (s)']:>7.1f}s")

bestes = max(ergebnisse, key=lambda x: x["Test-Acc"])
print(f"\nBeste Konfiguration: {bestes['Name']} → Acc: {bestes['Test-Acc']:.4f}")

# ── 6. Visualisierung ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Genauigkeit vs. Zeit
akk   = [r["Test-Acc"]  for r in ergebnisse]
zeiten = [r["Zeit (s)"] for r in ergebnisse]
namen  = [r["Name"]     for r in ergebnisse]

axes[0].scatter(zeiten, akk, s=100, c=range(len(ergebnisse)), cmap="tab10")
for i, name in enumerate(namen):
    axes[0].annotate(name, (zeiten[i], akk[i]), fontsize=7,
                     xytext=(3, 3), textcoords="offset points")
axes[0].set_title("Test-Genauigkeit vs. Trainingszeit")
axes[0].set_xlabel("Trainingszeit (s)")
axes[0].set_ylabel("Test-Genauigkeit")
axes[0].grid(True)

# Lernkurven
for name, val_accs in verlauf_alle.items():
    axes[1].plot(val_accs, label=name, alpha=0.8)
axes[1].set_title("Validierungs-Genauigkeit aller Varianten")
axes[1].set_xlabel("Epoche")
axes[1].set_ylabel("Genauigkeit")
axes[1].legend(fontsize=6)
axes[1].grid(True)

plt.suptitle("CNN Hyperparameter-Vergleich (MNIST, 2000 Samples)", fontsize=13)
plt.tight_layout()
plt.savefig("F7_3_hyperparameter_vergleich.png", dpi=100)
plt.show()
print("Diagramm gespeichert: F7_3_hyperparameter_vergleich.png")
